# So sánh LSTM (baseline) và GAT-GRU

**Câu hỏi:** đưa đồ thị phụ thuộc dịch vụ (GAT) vào có giúp dự báo nhu cầu tải/tài nguyên tốt hơn một mô hình chuỗi thời gian không có đồ thị (LSTM) không?

**Điều kiện công bằng** (được đảm bảo bởi `modeling-common/`):
- cùng run và cùng split (`splits.json`), nên test trên **cùng các run**;
- cùng cách tạo feature nút, cùng chuẩn hóa, cùng target, cùng horizon;
- cùng quy trình feature selection;
- cùng ngân sách Optuna (`ft.BUDGET`), cùng loss, optimizer và early stopping;
- cùng số seed khi đánh giá cuối.

Mỗi mô hình có thể chọn window w khác nhau, nên các cửa sổ test được **ghép theo (run, bước dự báo đầu tiên)**.

**Chỉ số:**

| Chỉ số | Ý nghĩa cho autoscaling |
|---|---|
| MAE, RMSE (đơn vị gốc) | Độ chính xác chung (theo đề cương) |
| MAE theo khoảng dự báo | Chất lượng dự báo xa: quyết định khả năng scale *trước* tải |
| Sai số dự báo thiếu (under) / thừa (over) | Dự báo thiếu: không đủ replica, vi phạm SLO. Dự báo thừa: lãng phí tài nguyên |
| Sai số tại đầu spike | Đúng thời điểm proactive scaling có giá trị nhất |
| Thắng/thua theo run + sign test | Các cửa sổ chồng lấn nên không độc lập; đơn vị độc lập là **run** |
| Số tham số, thời gian suy luận | Chi phí khi đưa vào controller |

**Chọn bộ dataset** bằng biến `DATASET` (đặt **giống nhau** ở mọi notebook của một lần thí nghiệm):

| `DATASET` | Nguồn | Kịch bản | Split | Thư mục đầu ra |
|---|---|---|---|---|
| `'load-testing'` | `load-testing/results/` (Locust, mô hình đóng) | normal, spike, bursty | `modeling-common/splits.json` | `processed/` |
| `'math'` | `math-load-testing/results/` (mô hình toán, luồng đến mở) | nhpp, mmpp, pareto, onoff | `modeling-common/splits_math.json` | `processed_math/` |

Hai bộ có split và thư mục kết quả riêng nên không ảnh hưởng nhau.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

COMMON = (Path('..') / 'modeling-common').resolve()   # code dùng chung cho GAT-GRU và LSTM
sys.path.insert(0, str(COMMON))
import forecast_data as fd

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 180)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})

import forecast_metrics as fm

In [ ]:
DATASET = 'load-testing'          # 'load-testing' | 'math' - giống các notebook trước
TAG = '_math' if DATASET == 'math' else ''
ONSET_SCENARIO = 'spike' if DATASET == 'load-testing' else 'mmpp'   # kịch bản có tải tăng đột ngột


def find(folder, model):
    for sub in (f'processed{TAG}', f'processed{TAG}_demo'):
        p = Path(folder) / sub / f'{model}_predictions_test.npz'
        if p.exists():
            return p
    raise FileNotFoundError(f'Chưa có {model}_predictions_test.npz: chạy train_{model}.ipynb trước')


paths = {'LSTM': find('.', 'lstm'), 'GAT-GRU': find('../load-testing', 'gatgru')}
raw = {k: fm.load_predictions(p) for k, p in paths.items()}
if raw['LSTM']['meta'].get('demo') or raw['GAT-GRU']['meta'].get('demo'):
    print('⚠️ Đang so sánh trên DỮ LIỆU DEMO, kết quả không có giá trị khoa học')
lstm, gat = fm.align(raw['LSTM'], raw['GAT-GRU'])
print({k: len(v['info']) for k, v in raw.items()}, '-> cửa sổ chung:', len(lstm['info']))
assert np.allclose(lstm['Y_true'], gat['Y_true'], atol=1e-3), 'Ground truth khác nhau: hai mô hình không dùng cùng dữ liệu!'
SERVICES, TARGETS = fd.SERVICES, fd.TARGETS

## 1. Tổng quan (trung bình ± độ lệch chuẩn qua các seed)

In [ ]:
rows = []
for name, d in [('LSTM', lstm), ('GAT-GRU', gat)]:
    rows.append(fm.seed_summary(d, TARGETS, name)[0])
pers = fm.error_table(fm.persistence(lstm), lstm['Y_true'], TARGETS).round(4).astype(str)
pers.insert(0, 'model', 'persistence')
table = pd.concat(rows + [pers])
table

In [ ]:
_, m_l, _ = fm.seed_summary(lstm, TARGETS, 'LSTM')
_, m_g, _ = fm.seed_summary(gat, TARGETS, 'GAT-GRU')
improve = (1 - m_g / m_l) * 100
print('Mức cải thiện của GAT-GRU so với LSTM (%, dương = GAT-GRU tốt hơn):')
improve.round(1)

## 2. Theo khoảng dự báo

In [ ]:
h = np.arange(1, fd.HORIZON + 1) * fd.STEP_SEC
fig, axes = plt.subplots(1, len(TARGETS), figsize=(13, 4))
for k, t in enumerate(TARGETS):
    for name, d, c in [('LSTM', lstm, 'C0'), ('GAT-GRU', gat, 'C3')]:
        e = np.stack([fm.by_horizon(d['Y_pred'][s], d['Y_true'], k) for s in range(len(d['Y_pred']))])
        axes[k].errorbar(h, e.mean(0), yerr=e.std(0), marker='o', capsize=3, color=c, label=name)
    axes[k].plot(h, fm.by_horizon(fm.persistence(lstm), lstm['Y_true'], k), 's--', color='gray', label='persistence')
    axes[k].set_xlabel('dự báo trước (giây)')
    axes[k].set_ylabel(f'MAE {t}')
    axes[k].legend()
fig.tight_layout()
plt.show()

## 3. Theo kịch bản và theo service

In [ ]:
mean_pred = lambda d: d['Y_pred'].mean(0)     # dự báo trung bình qua seed
out = {}
for k, t in enumerate(TARGETS):
    for name, d in [('LSTM', lstm), ('GAT-GRU', gat)]:
        out[(t, name)] = fm.by_group(mean_pred(d), d['Y_true'], d['info'], k, col=1)
pd.DataFrame(out).round(4)

In [ ]:
fig, axes = plt.subplots(1, len(TARGETS), figsize=(15, 4.5))
x = np.arange(len(SERVICES))
for k, t in enumerate(TARGETS):
    for j, (name, d) in enumerate([('LSTM', lstm), ('GAT-GRU', gat)]):
        axes[k].bar(x + (j - 0.5) * 0.4, fm.by_service(mean_pred(d), d['Y_true'], k, SERVICES), 0.4, label=name)
    axes[k].set_xticks(x, SERVICES, rotation=70, fontsize=8)
    axes[k].set_title(f'MAE {t} theo service')
    axes[k].legend()
fig.tight_layout()
plt.show()

## 4. Dự báo thiếu / thừa và đầu spike
Trong autoscaling, **dự báo thiếu** nguy hiểm hơn dự báo thừa vì gây vi phạm SLO. Cửa sổ *đầu spike* là các cửa sổ (kịch bản `spike`, hoặc `mmpp` với bộ toán học) mà tải frontend tương lai vượt 1,5 lần giá trị quan sát cuối.

In [ ]:
onset = fm.spike_onset_mask(lstm, SERVICES, scenario=ONSET_SCENARIO)
rows = []
for name, d in [('LSTM', lstm), ('GAT-GRU', gat), ('persistence', None)]:
    pred = fm.persistence(lstm) if d is None else mean_pred(d)
    ref = lstm
    for subset, mask in [('tất cả', slice(None)), ('đầu spike', onset)]:
        if subset == 'đầu spike' and not onset.any():
            continue
        e = fm.error_table(pred[mask], ref['Y_true'][mask], TARGETS)
        for t in TARGETS:
            rows.append({'model': name, 'tập': subset, 'target': t, **e.loc[t].to_dict()})
print('Số cửa sổ đầu spike:', int(onset.sum()))
pd.DataFrame(rows).set_index(['tập', 'target', 'model']).sort_index().round(4)

## 5. Kiểm định theo run
Mỗi run test là một đơn vị độc lập. Đếm số run GAT-GRU có MAE thấp hơn LSTM, rồi dùng **sign test** hai phía. Với ít run test, p-value lớn là bình thường; hãy dùng kiểm định chéo theo run (xem `DATASET_GUIDE.md`) để có thêm điểm dữ liệu.

In [ ]:
rows = []
for k, t in enumerate(TARGETS):
    a = fm.by_group(mean_pred(lstm), lstm['Y_true'], lstm['info'], k, col=0)
    b = fm.by_group(mean_pred(gat), gat['Y_true'], gat['info'], k, col=0)
    wins = int((b < a).sum())
    rows.append({'target': t, 'số run test': len(a), 'GAT-GRU thắng': wins, 'LSTM thắng': int((a < b).sum()),
                 'sign test p': fm.sign_test(wins, len(a))})
pd.DataFrame(rows).set_index('target')

## 6. Chi phí mô hình

In [ ]:
pd.DataFrame({name: {'tham số': d['meta']['n_params'], 'suy luận 1 cửa sổ (ms)': round(d['meta']['inference_ms'], 2),
                     'window w': d['meta']['config']['window'], 'feature nút': len(d['meta']['node_features']),
                     'feature cạnh': len(d['meta'].get('edge_features', [])),
                     'val MAE (scaled)': round(float(np.mean(d['meta']['val_mae_scaled'])), 4)}
              for name, d in raw.items()})

## 7. Đọc kết quả

- GAT-GRU **tốt hơn rõ** ở các service downstream (productcatalog, currency, cart) và ở cửa sổ đầu spike thì thông tin đồ thị có giá trị: tải ở frontend được lan truyền xuống qua các cạnh *trước khi* downstream thấy tải.
- Hai mô hình **ngang nhau** thì nguyên nhân có thể là: tải các service tương quan quá chặt (một chuỗi là đủ), cạnh/feature cạnh bị rơi span, hoặc mô hình `joint` của LSTM đã học được tương quan chéo. Khi đó xem `use_edge_features`/`edge_direction` trong cấu hình tốt nhất và ablation trong `feature_selection_gatgru`.
- Lựa chọn mô hình cho controller cần cân nhắc cả **dự báo thiếu** (rủi ro SLO) lẫn chi phí suy luận, không chỉ MAE trung bình.